# ASOS PRCP Comparison: Minute vs Daily Data

Compare precipitation from minute-resolution ASOS data vs daily NOAA data.
- **Minute data**: `precip_mm` summed to daily totals
- **Daily data**: `PRCP` column (should match minute sums)

Both should be in **mm**.

In [2]:
# Cell 1: Setup
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
import sys
import os
warnings.filterwarnings('ignore')

# Add current directory to path for imports
# Notebook is in: src/analysis/asos_analysis/
# asos_helperes.py is in the same directory
notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
# For Jupyter notebooks, find directory containing asos_helperes.py
current_dir = Path.cwd()
if (current_dir / 'asos_helperes.py').exists():
    notebook_dir = current_dir
elif (current_dir.parent / 'asos_helperes.py').exists():
    notebook_dir = current_dir.parent
else:
    # Fallback: try to find from common notebook locations
    for possible_dir in [Path.cwd(), Path.cwd().parent]:
        if (possible_dir / 'asos_helperes.py').exists():
            notebook_dir = possible_dir
            break

if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from asos_helperes import (
    STATIONS, load_minute_data, load_daily_data,
    aggregate_minute_to_daily, merge_min_daily,
    summary_stats, plot_accumulated, plot_daily_bars
)

# CONFIG - Adjust paths as needed
# From src/analysis/asos_analysis/ go up 2 levels to src/, then to data/noaa_asos/
DATA_DIR = notebook_dir.parent.parent / 'data' / 'noaa_asos'
DAILY_FILE = DATA_DIR / 'snow' / 'snow_data_processed.csv'

# Verify paths exist
if not DATA_DIR.exists():
    print(f"⚠ Warning: DATA_DIR does not exist: {DATA_DIR}")
if not DAILY_FILE.exists():
    print(f"⚠ Warning: DAILY_FILE does not exist: {DAILY_FILE}")

# Analysis period (None = all data)
START_DATE = None  # e.g., '2024-01-01'
END_DATE = None    # e.g., '2024-03-31'

print(f'✓ Setup complete')
print(f'  Notebook dir: {notebook_dir}')
print(f'  DATA_DIR: {DATA_DIR}')
print(f'  DAILY_FILE: {DAILY_FILE}')

ImportError: cannot import name 'STATIONS' from 'asos_helperes' (/Users/drorjac/OpenMesh_pynncml/src/analysis/asos_analysis/asos_helperes.py)

In [ ]:
# Cell 2: Load data
print('Loading minute data...')
df_min = load_minute_data(DATA_DIR)

print('\nLoading daily data...')
df_daily = load_daily_data(DAILY_FILE)

# Quick unit check
print('\n--- Unit Check ---')
print(f"Minute precip_mm: min={df_min['precip_mm'].min():.4f}, max={df_min['precip_mm'].max():.4f}")
print(f"Daily PRCP:       min={df_daily['PRCP'].min():.4f}, max={df_daily['PRCP'].max():.4f}")

In [ ]:
# Cell 3: Process - aggregate minute to daily, merge with daily
df_min_daily = aggregate_minute_to_daily(df_min)
print(f'Aggregated minute data: {len(df_min_daily)} station-days')

df_merged = merge_min_daily(df_min_daily, df_daily, START_DATE, END_DATE)
print(f'Merged data: {len(df_merged)} rows')

# Show sample
print('\nSample (first rainy days):')
sample = df_merged[(df_merged['PRCP_min'] > 0) | (df_merged['PRCP_daily'] > 0)].head(10)
print(sample[['DATE', 'station_code', 'PRCP_min', 'PRCP_daily', 'diff', 'n_records']].to_string(index=False))

In [ ]:
# Cell 4: Summary statistics
stats = summary_stats(df_merged)
print('\n=== Summary Statistics per Station ===')
print(stats.to_string(index=False))

In [ ]:
# Cell 5: Accumulated PRCP plot
fig, axes = plot_accumulated(df_merged)
plt.show()

In [ ]:
# Cell 6: Daily bar comparison (rainy days only)
# Optionally set a shorter period for better visibility
BAR_START = '2024-01-01'  # Adjust as needed
BAR_END = '2024-01-31'    # Adjust as needed

fig, axes = plot_daily_bars(df_merged, start_date=BAR_START, end_date=BAR_END)
plt.show()

In [ ]:
# Cell 7: Investigate discrepancies
# Large differences might indicate:
# 1. Unit mismatch (one in inches, one in mm)
# 2. Missing minute data (check n_records column)
# 3. Different measurement times (daily 00:00-24:00 vs other)

# Find days with biggest discrepancies
big_diff = df_merged[df_merged['diff'].abs() > 10].sort_values('diff', key=abs, ascending=False)
print('Days with |diff| > 10 mm:')
print(big_diff[['DATE', 'station_code', 'PRCP_min', 'PRCP_daily', 'diff', 'n_records']].head(20).to_string(index=False))

In [ ]:
# Cell 8: Check if daily data is in inches (common NOAA format)
# If daily PRCP is in inches, multiply by 25.4 to convert to mm

# Test: if daily values are typically 10x smaller, they might be in inches
rainy = df_merged[(df_merged['PRCP_min'] > 0) & (df_merged['PRCP_daily'] > 0)]
if len(rainy) > 0:
    ratio = rainy['PRCP_min'].mean() / rainy['PRCP_daily'].mean()
    print(f'Mean ratio (minute/daily): {ratio:.2f}')
    if 20 < ratio < 30:
        print('⚠ Ratio ~25.4 suggests daily PRCP might be in INCHES!')
        print('  Try: df_daily["PRCP"] = df_daily["PRCP"] * 25.4')